# CataLIST Stage 2 — Google Colab Setup

This notebook sets up the CataLIST model in Google Colab for testing and inference.

**Setup time:** ~30 seconds (first run)  
**Requirements:** None (everything installed automatically)  
**GPU:** Optional (works on CPU too, slower)

---

## Step 1: Clone Repository & Install Dependencies

Run this cell first.

In [ ]:
import sys
import subprocess
import os
from pathlib import Path

def run_cmd(cmd: str, silent=False) -> str:
    """Run shell command."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0 and not silent:
        print(f"Error: {result.stderr}")
    return result.stdout

print("="*60)
print("CataLIST Stage 2 — Colab Setup")
print("="*60)

# ============ STEP 1: Clone Repositories ============
print("\n[1/6] Cloning/updating repositories...")

# Clone main repo
repo_path = Path("/content/eumine_databridge_2026")
if repo_path.exists():
    print(f"  ✓ Main repo exists, pulling latest...")
    run_cmd("cd /content/eumine_databridge_2026 && git pull", silent=True)
else:
    print(f"  Cloning main repo...")
    run_cmd("git clone https://github.com/Iman-Peivaste/eumine_databridge_2026.git /content/eumine_databridge_2026")
    print(f"  ✓ Cloned to {repo_path}")

# Clone matfed-api-template
matfed_path = Path("/content/matfed-api-template")
if matfed_path.exists():
    print(f"  ✓ matfed-api-template already exists")
else:
    print(f"  Cloning matfed-api-template...")
    run_cmd("git clone --depth 1 https://github.com/EuMINe-COST/eumine_hackathon_2026.git /tmp/eumine_hackathon_2026")
    run_cmd("mv /tmp/eumine_hackathon_2026/matfed-api-template /content/matfed-api-template")
    print(f"  ✓ Extracted to {matfed_path}")

# ============ STEP 2: Add to Python Path ============
print("\n[2/6] Setting up Python paths...")
sys.path.insert(0, str(repo_path / "src"))
sys.path.insert(0, str(matfed_path))
print(f"  ✓ Added to sys.path:")
print(f"    - {repo_path / 'src'}")
print(f"    - {matfed_path}")

# ============ STEP 3: Install Package (editable mode) ============
print("\n[3/6] Installing eumine_databridge package...")
# First, uninstall if it exists from previous attempts
run_cmd("pip uninstall -y eumine_databridge", silent=True)
# Install fresh in editable mode
run_cmd(f"pip install -q -e {repo_path}")
print(f"  ✓ Package installed in editable mode")

# ============ STEP 4: Install Dependencies ============
print("\n[4/6] Installing dependencies...")
packages = [
    "pymatgen==2025.6.14",
    "torch==2.4.0",
    "torch-geometric",
    "alignn==2026.4.2",
    "mace-torch==0.3.15",
    "optuna>=3.6",
    "pandas",
    "numpy",
]

for pkg in packages:
    pkg_name = pkg.split("==")[0].split(">=")[0]
    try:
        __import__(pkg_name.replace("-", "_"))
        print(f"  ✓ {pkg_name}")
    except ImportError:
        print(f"  Installing {pkg_name}...")
        run_cmd(f"pip install -q {pkg}", silent=True)
        print(f"    ✓ Installed")

# ============ STEP 5: Verify Key Imports ============
print("\n[5/6] Verifying key imports...")

# Force reimport
import importlib
import sys

# Remove cached modules
for module in list(sys.modules.keys()):
    if 'eumine' in module or 'matfed' in module:
        del sys.modules[module]

try:
    from eumine_databridge.matfed.predictor import LISTEuMINePredictor
    print(f"  ✓ LISTEuMINePredictor imported")

    from eumine_databridge.matfed.federation import FederatedEnsemble
    print(f"  ✓ FederatedEnsemble imported")

    from matfed_api.predictor import MatFedPredictor
    print(f"  ✓ MatFedPredictor (matfed_api) imported")
    
    print(f"\n✓ All user-facing APIs loaded successfully!")
except Exception as e:
    print(f"  ✗ Import error: {e}")
    import traceback
    traceback.print_exc()

# ============ STEP 6: Summary ============
print("\n[6/6] Setup summary:")
print(f"  Repository: {repo_path}")
print(f"  MatFed API: {matfed_path}")
print(f"  Python package: eumine_databridge (editable mode)")

print("\n" + "="*60)
print("✓ Setup complete! Ready to use.")
print("="*60)
print("\nNext: Run Cell 3 to initialize the predictor.")

## Step 2: Mount Google Drive (Optional)

If you have model artifacts in Drive, mount it here.

In [ ]:
from google.colab import drive

print("Mounting Google Drive...")
drive.mount("/content/gdrive")
print("✓ Drive mounted at /content/gdrive")
print("\nYou can now access files like:")
print("  /content/gdrive/My Drive/my_models/")

## Step 3: Initialize Predictor

Load the model (this assumes artifacts are in the repo or Drive).

In [ ]:
from eumine_databridge.matfed.predictor import LISTEuMINePredictor
from pathlib import Path

print("="*60)
print("Initializing CataLIST Predictor")
print("="*60)

repo_path = Path("/content/eumine_databridge_2026")
model_dir = repo_path / "models" / "combined_retrain"

print(f"\nModel directory: {model_dir}")
print(f"Model artifacts exist: {model_dir.exists()}")

# Initialize predictor WITHOUT auto-loading (to avoid errors when model artifacts missing)
print(f"\nInitializing predictor (interface-only mode)...")
predictor = LISTEuMINePredictor()
predictor._loaded = False  # Mark as not loaded
print(f"✓ Predictor initialized")

# Try to load model if artifacts exist
if model_dir.exists():
    print(f"\n✓ Model artifacts found!")
    print(f"Loading model...")
    try:
        predictor.load_model(str(model_dir))
        print(f"✓ Model loaded successfully!")
        
        desc = predictor.describe()
        print(f"\n" + "-"*60)
        print(f"Model Information:")
        print(f"-"*60)
        print(f"  Team: {desc['team_name']}")
        print(f"  Version: {desc['model_version']}")
        print(f"  Type: {desc['model_type']}")
        print(f"  Performance (OOF): {desc['performance'].get('oof_total_score_40pts', 'N/A')}/40")
        print(f"-"*60)
    except Exception as e:
        print(f"✗ Error loading model: {e}")
        print(f"  Continuing in interface-only mode...")
else:
    print(f"\n⚠ Model artifacts not found (expected!)")
    print(f"\nℹ The 14.3 GB model files are stored locally, not in GitHub.")
    print(f"  Options:")
    print(f"    1. Ask for download link: euminecost@gmail.com")
    print(f"    2. Mount Google Drive: see Cell 2 (optional)")
    print(f"    3. Use interface-only mode (predictions won't work, but API works)")

print(f"\n✓ Predictor ready for predictions!")
print(f"  - Use predictor.predict([structures]) to make predictions")
print(f"  - Use predictor.describe() to see model info")
print(f"  - Use federation interface in Cell 5 for Stage 2")

## Step 4: Test Prediction (Dummy Structures)

Create and predict on sample structures.

In [ ]:
from pymatgen.core import Structure, Lattice
import pandas as pd

# Create sample structures
structures = []
names = []

# Silicon (diamond cubic)
lat = Lattice.cubic(5.4)
si = Structure(lat, ["Si", "Si"], [[0.0, 0.0, 0.0], [0.25, 0.25, 0.25]])
structures.append(si)
names.append("Si (diamond)")

# NaCl (rock salt)
lat = Lattice.cubic(5.64)
nacl = Structure(lat, ["Na", "Cl"], [[0.0, 0.0, 0.0], [0.5, 0.5, 0.5]])
structures.append(nacl)
names.append("NaCl (rock salt)")

# GaAs (zinc blende)
lat = Lattice.cubic(5.65)
gaas = Structure(lat, ["Ga", "As"], [[0.0, 0.0, 0.0], [0.25, 0.25, 0.25]])
structures.append(gaas)
names.append("GaAs (zinc blende)")

print(f"Created {len(structures)} test structures:\n")
for name, s in zip(names, structures):
    print(f"  • {name}")
    print(f"    Composition: {s.composition.reduced_formula}")
    print(f"    Atoms: {len(s)}")

print(f"\nMaking predictions...\n")

if predictor._loaded:
    predictions = predictor.predict(structures)
    
    # Display results
    results = []
    for name, pred in zip(names, predictions):
        results.append({
            "Structure": name,
            "Formation Energy (eV/atom)": f"{pred['formation_energy_per_atom']:.4f}",
            "Band Gap (eV)": f"{pred['band_gap']:.4f}",
            "σ_EF": f"{pred['uncertainty_ef']:.4f}",
            "σ_BG": f"{pred['uncertainty_bg']:.4f}",
        })
    
    df = pd.DataFrame(results)
    print(df.to_string(index=False))
    print(f"\n✓ Predictions complete!")
else:
    print("⚠ Predictor not loaded (model artifacts missing).")
    print("Use Step 2 to mount Google Drive and load artifacts.")

## Step 5: Federation Test (For Cluj)

Test the federation interface with mock partners.

In [ ]:
from eumine_databridge.matfed.federation import FederatedEnsemble
import numpy as np

print("Testing federation interface...\n")

# Create federation
fed = FederatedEnsemble()
fed.add_predictor(predictor, "CataLIST")

print(f"✓ FederatedEnsemble initialized")
print(f"  Predictors: {list(fed.predictors.keys())}")
print(f"\nIn Cluj, you would add partner models here:")
print(f"""\n  # Load TakeMe2Romania
  from their_package import TheirPredictor
  t2r = TheirPredictor()
  t2r.load_model("/path/to/their/model")
  fed.add_predictor(t2r, "TakeMe2Romania")
  
  # Organizers provide calibration data
  cal_structures, cal_ef, cal_bg = load_calibration_data()
  
  # Fit federation
  result = fed.fit(cal_structures, cal_ef, cal_bg, n_trials=200)
  
  # Predict on test set
  predictions = fed.predict(test_structures)
""")

print(f"✓ Federation interface ready!")

## Step 6: Use Your Own Structures

Upload a CIF file and predict on it.

In [ ]:
from google.colab import files
from pymatgen.core import Structure

print("Upload CIF file(s) to predict on...\n")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"Processing {filename}...")
    
    try:
        # Parse structure
        s = Structure.from_file(filename)
        print(f"  Composition: {s.composition.reduced_formula}")
        print(f"  Atoms: {len(s)}")
        
        # Predict
        if predictor._loaded:
            pred = predictor.predict([s])[0]
            print(f"  Formation Energy: {pred['formation_energy_per_atom']:.4f} eV/atom")
            print(f"  Band Gap: {pred['band_gap']:.4f} eV")
            print(f"  Uncertainty EF: ±{pred['uncertainty_ef']:.4f}")
            print(f"  Uncertainty BG: ±{pred['uncertainty_bg']:.4f}")
        else:
            print(f"  ⚠ Predictor not loaded")
    except Exception as e:
        print(f"  ✗ Error: {e}")
    
    print()

## Notes

- **Model artifacts:** The 14.3 GB model files are not in the repo. Download them from:
  - Google Drive link: (contact Iman)
  - Or mount Drive and access them there

- **GPU:** Colab's free GPU (T4) is fast enough for predictions
  - Single structure: ~1-2 seconds
  - Batch of 10: ~5-10 seconds
  - CPU mode works too (slower)

- **For Stage 2 in Cluj:** This notebook helps you test before the sprint
  - Verify your model loads
  - Test federation interface
  - Practice loading partner models

---

**Questions?** Contact: euminecost@gmail.com
